# SQL Joins: Complete Guide with Sequencing QC Data

This notebook covers every major JOIN type using the `sequencing_qc` PostgreSQL database.
The same four-table schema from `sql_warmups.ipynb` is used:

```
patients  →  samples  →  sequencing_runs  →  qc_metrics
```

To make outer joins meaningful, a few **orphan rows** are added at setup:
- Patients with no samples
- Samples with no sequencing run
- A sequencing run with no QC metrics

---

## Join Types Covered
| # | Join Type | When to Use |
|---|-----------|-------------|
| 1 | INNER JOIN | Only rows that match on both sides |
| 2 | LEFT JOIN | All rows from the left table; NULLs where no match |
| 3 | RIGHT JOIN | All rows from the right table; NULLs where no match |
| 4 | FULL OUTER JOIN | All rows from both tables; NULLs on either side |
| 5 | CROSS JOIN | Every combination (Cartesian product) |
| 6 | SELF JOIN | A table joined to itself |
| 7 | Multi-table JOIN | Chaining three or more joins |
| 8 | JOIN with USING | Shorthand when key column names match |
| 9 | Non-equi JOIN | Join on a range or inequality condition |
| 10 | Derived Table JOIN | JOIN to a subquery result |
| 11 | Anti-JOIN | Rows in A with **no** match in B (LEFT JOIN + IS NULL) |
| 12 | Semi-JOIN | Rows in A that **have** a match in B (EXISTS) |
| 13 | NATURAL JOIN | Auto-join on identically-named columns |

## 1. Setup and Database Connection

In [ ]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine, text

# --- connection parameters (match sql_warmups.ipynb) ---
DB_USER = 'shanebrubaker'
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'sequencing_qc'

engine = create_engine(f'postgresql://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

def get_conn():
    return psycopg2.connect(dbname=DB_NAME, user=DB_USER, host=DB_HOST, port=DB_PORT)

def run(query, **kwargs):
    """Helper: execute a SQL string and return a DataFrame."""
    return pd.read_sql(query, engine, **kwargs)

# Quick connectivity check
with engine.connect() as c:
    print('Connected to', DB_NAME)

## 2. Add Orphan Rows for Outer-Join Demonstrations

Real-world data always has incomplete records. We insert a few here so outer joins
return visibly different results than inner joins.

In [ ]:
conn = get_conn()
cur = conn.cursor()

# Remove any orphans from a previous run of this notebook
cur.execute("DELETE FROM qc_metrics   WHERE run_id  = 'RUN9999';")
cur.execute("DELETE FROM sequencing_runs WHERE run_id = 'RUN9999';")
cur.execute("DELETE FROM samples       WHERE sample_id IN ('S9901','S9902');")
cur.execute("DELETE FROM patients      WHERE patient_id IN ('PT9901','PT9902');")

# Patient with NO samples
cur.execute("""
    INSERT INTO patients (patient_id, age, gender, diagnosis, enrollment_date)
    VALUES
        ('PT9901', 55, 'Female', 'Melanoma',   '2024-08-01'),
        ('PT9902', 48, 'Male',   'Glioblastoma','2024-09-15');
""")

# Sample with NO sequencing run (belongs to an existing patient)
cur.execute("""
    INSERT INTO samples (sample_id, patient_id, sample_type, collection_date, tissue_source, dna_concentration)
    VALUES ('S9901', 'PT0001', 'Tumor', '2024-11-01', 'Breast', 88.5);
""")

# Sequencing run with NO qc_metrics entry (belongs to an existing sample)
cur.execute("""
    INSERT INTO sequencing_runs (run_id, sample_id, sequencer, run_date, platform, library_prep)
    VALUES ('RUN9999', 'S0001', 'NovaSeq6000', '2024-12-01', 'Illumina', 'TruSeq DNA');
""")

conn.commit()
cur.close()
conn.close()

print('Orphan rows inserted.')
print('  Patients with no samples : PT9901, PT9902')
print('  Sample  with no run      : S9901')
print('  Run     with no QC entry : RUN9999')

---
## 3. INNER JOIN

Returns **only rows that have a matching value in both tables**. Unmatched rows on
either side are dropped.

```
patients  ┤├  samples
          (only patients that have ≥1 sample appear)
```

In [ ]:
# INNER JOIN: patients that have at least one sample
# PT9901 and PT9902 (no samples) will NOT appear.
query = """
SELECT p.patient_id, p.diagnosis, p.age,
       s.sample_id, s.sample_type
FROM   patients p
INNER JOIN samples s ON p.patient_id = s.patient_id
ORDER BY p.patient_id, s.sample_id;
"""
df = run(query)
print(f'Rows returned: {len(df)}  (all patients have ≥1 sample; PT9901/PT9902 excluded)')
display(df)

In [ ]:
# INNER JOIN across three tables: only runs that also have QC metrics
# RUN9999 (no qc_metrics row) will NOT appear.
query = """
SELECT sr.run_id, sr.sequencer,
       qm.mean_coverage, qm.mean_quality_score, qm.pass_filter
FROM   sequencing_runs sr
INNER JOIN qc_metrics qm ON sr.run_id = qm.run_id
ORDER BY sr.run_id;
"""
df = run(query)
print(f'Rows returned: {len(df)}  (RUN9999 excluded — no QC entry)')
display(df)

---
## 4. LEFT JOIN (LEFT OUTER JOIN)

Returns **all rows from the LEFT table**, plus matched rows from the right table.
Where there is no match, right-table columns are `NULL`.

```
patients  ⟕  samples
         (all patients; NULL sample columns when no sample exists)
```

In [ ]:
# LEFT JOIN: all patients, with sample info where it exists
query = """
SELECT p.patient_id, p.diagnosis,
       s.sample_id, s.sample_type
FROM   patients p
LEFT JOIN samples s ON p.patient_id = s.patient_id
ORDER BY p.patient_id, s.sample_id;
"""
df = run(query)
print(f'Rows returned: {len(df)}  (PT9901 and PT9902 appear with NULL sample columns)')
display(df)

In [ ]:
# LEFT JOIN: all sequencing runs, with QC metrics where available
# RUN9999 (no qc_metrics row) appears with NULL metrics.
query = """
SELECT sr.run_id, sr.sequencer, sr.run_date,
       qm.mean_coverage, qm.pass_filter
FROM   sequencing_runs sr
LEFT JOIN qc_metrics qm ON sr.run_id = qm.run_id
ORDER BY sr.run_id;
"""
df = run(query)
print(f'Rows returned: {len(df)}  (RUN9999 appears with NULL metrics)')
display(df)

---
## 5. RIGHT JOIN (RIGHT OUTER JOIN)

Mirror image of LEFT JOIN. Returns **all rows from the RIGHT table**, plus matched
rows from the left table. Where there is no match, left-table columns are `NULL`.

> Tip: a RIGHT JOIN can always be rewritten as a LEFT JOIN by swapping the tables.
> Most teams standardize on LEFT JOIN for readability.

In [ ]:
# RIGHT JOIN: all samples, with patient info where it exists.
# In this schema every sample has a patient_id FK, so no NULLs on the left here —
# but the query demonstrates the syntax and direction clearly.
query = """
SELECT p.patient_id, p.diagnosis,
       s.sample_id, s.sample_type, s.dna_concentration
FROM   patients p
RIGHT JOIN samples s ON p.patient_id = s.patient_id
ORDER BY s.sample_id;
"""
df = run(query)
print(f'Rows returned: {len(df)}')
display(df)

In [ ]:
# Equivalent rewrite as a LEFT JOIN (swap table order)
query = """
SELECT p.patient_id, p.diagnosis,
       s.sample_id, s.sample_type, s.dna_concentration
FROM   samples s
LEFT JOIN patients p ON p.patient_id = s.patient_id
ORDER BY s.sample_id;
"""
df = run(query)
print(f'Same result, written as LEFT JOIN: {len(df)} rows')
display(df)

---
## 6. FULL OUTER JOIN

Returns **all rows from both tables**. Where a row in either table has no match,
the other table's columns are `NULL`.

```
patients  ⟗  samples
         (all patients AND all samples; NULLs on whichever side has no match)
```

In [ ]:
# FULL OUTER JOIN: every patient and every sample, matched where possible
query = """
SELECT p.patient_id, p.diagnosis,
       s.sample_id, s.sample_type
FROM   patients p
FULL OUTER JOIN samples s ON p.patient_id = s.patient_id
ORDER BY p.patient_id NULLS LAST, s.sample_id;
"""
df = run(query)
print(f'Rows: {len(df)}  (includes PT9901/PT9902 with NULL sample cols)')
display(df)

In [ ]:
# FULL OUTER JOIN: sequencing_runs vs qc_metrics
# RUN9999 appears (no QC row); all qc rows with a matching run also appear.
query = """
SELECT sr.run_id       AS sr_run_id,
       sr.sequencer,
       qm.run_id       AS qm_run_id,
       qm.mean_coverage,
       qm.pass_filter
FROM   sequencing_runs sr
FULL OUTER JOIN qc_metrics qm ON sr.run_id = qm.run_id
ORDER BY sr.run_id NULLS LAST;
"""
df = run(query)
print(f'Rows: {len(df)}  — RUN9999 has NULL qm_run_id and NULL metrics')
display(df)

---
## 7. CROSS JOIN (Cartesian Product)

Produces **every combination** of rows from the two tables. If table A has M rows
and table B has N rows, the result has M × N rows.

Common use: generating all possible combinations for a report or test matrix.

In [ ]:
# All combinations of distinct sequencer types × distinct diagnoses
# Useful for building a complete reporting matrix (including zero-count cells)
query = """
SELECT seq.sequencer, diag.diagnosis
FROM   (SELECT DISTINCT sequencer FROM sequencing_runs) seq
CROSS JOIN (SELECT DISTINCT diagnosis FROM patients) diag
ORDER BY seq.sequencer, diag.diagnosis;
"""
df = run(query)
print(f'Rows: {len(df)}  (3 sequencers × {len(df)//3} diagnoses)')
display(df)

In [ ]:
# Practical use: fill a coverage matrix with actual counts (including zeros)
query = """
WITH grid AS (
    SELECT seq.sequencer, diag.diagnosis
    FROM   (SELECT DISTINCT sequencer FROM sequencing_runs) seq
    CROSS JOIN (SELECT DISTINCT diagnosis FROM patients)   diag
),
actual AS (
    SELECT sr.sequencer, p.diagnosis, COUNT(*) AS run_count
    FROM   sequencing_runs sr
    JOIN   samples  s  ON sr.sample_id  = s.sample_id
    JOIN   patients p  ON s.patient_id  = p.patient_id
    GROUP  BY sr.sequencer, p.diagnosis
)
SELECT g.sequencer, g.diagnosis,
       COALESCE(a.run_count, 0) AS run_count
FROM   grid g
LEFT JOIN actual a USING (sequencer, diagnosis)
ORDER BY g.sequencer, g.diagnosis;
"""
df = run(query)
pivot = df.pivot(index='diagnosis', columns='sequencer', values='run_count').fillna(0).astype(int)
print('Run count matrix (0 = no runs for that combination):')
display(pivot)

---
## 8. SELF JOIN

A table joined **to itself** using two different aliases. Useful for comparing
rows within the same table — e.g., finding patients with the same diagnosis,
or samples from the same patient collected on different dates.

In [ ]:
# Find pairs of patients that share the same diagnosis
# p1.patient_id < p2.patient_id avoids duplicate pairs and self-matches
query = """
SELECT p1.patient_id AS patient_a,
       p2.patient_id AS patient_b,
       p1.diagnosis
FROM   patients p1
JOIN   patients p2
          ON  p1.diagnosis  = p2.diagnosis
          AND p1.patient_id < p2.patient_id
ORDER BY p1.diagnosis, p1.patient_id;
"""
df = run(query)
print(f'Patient pairs sharing a diagnosis: {len(df)} pairs')
display(df)

In [ ]:
# Find pairs of samples from the same patient (different sample types)
query = """
SELECT s1.patient_id,
       s1.sample_id   AS sample_a, s1.sample_type AS type_a,
       s2.sample_id   AS sample_b, s2.sample_type AS type_b
FROM   samples s1
JOIN   samples s2
          ON  s1.patient_id  = s2.patient_id
          AND s1.sample_id   < s2.sample_id
ORDER BY s1.patient_id, s1.sample_id;
"""
df = run(query)
print(f'Sample pairs from the same patient: {len(df)}')
display(df)

---
## 9. Multi-table JOIN (Chaining Three or More Tables)

Joins are applied left-to-right. Each successive JOIN adds another table to the
result set. Mix join types freely — e.g., INNER for required relationships,
LEFT for optional ones.

In [ ]:
# INNER JOIN across all four tables — the complete pipeline view
query = """
SELECT p.patient_id, p.diagnosis,
       s.sample_id,  s.sample_type,
       sr.run_id,    sr.sequencer,
       qm.mean_coverage, qm.mean_quality_score, qm.pass_filter
FROM   patients p
INNER JOIN samples         s  ON p.patient_id = s.patient_id
INNER JOIN sequencing_runs sr ON s.sample_id  = sr.sample_id
INNER JOIN qc_metrics      qm ON sr.run_id    = qm.run_id
ORDER BY p.patient_id, sr.run_id;
"""
df = run(query)
print(f'Complete pipeline rows: {len(df)}')
display(df)

In [ ]:
# Mixed: INNER for required joins, LEFT for optional QC metrics
# RUN9999 (no QC) now appears in the results with NULL metric columns.
query = """
SELECT p.patient_id, p.diagnosis,
       s.sample_id,  s.sample_type,
       sr.run_id,    sr.sequencer, sr.run_date,
       qm.mean_coverage, qm.pass_filter
FROM   patients p
INNER JOIN samples         s  ON p.patient_id = s.patient_id
INNER JOIN sequencing_runs sr ON s.sample_id  = sr.sample_id
LEFT  JOIN qc_metrics      qm ON sr.run_id    = qm.run_id
ORDER BY p.patient_id, sr.run_id;
"""
df = run(query)
print(f'Rows (includes runs without QC): {len(df)}')
print('RUN9999 entry:')
display(df[df['run_id'] == 'RUN9999'])
print('\nAll rows:')
display(df)

---
## 10. JOIN with USING

When the join key has the **same name in both tables**, `USING (column)` is a
shorthand for `ON a.column = b.column`. It also deduplicates the key column
in `SELECT *` output.

In [ ]:
# JOIN with USING: sequencing_runs and qc_metrics both have run_id
query = """
SELECT run_id, sequencer, run_date,
       mean_coverage, mean_quality_score, pass_filter
FROM   sequencing_runs
INNER JOIN qc_metrics USING (run_id)
ORDER BY run_id;
"""
df = run(query)
print('USING (run_id) — run_id appears only once in SELECT *:')
display(df)

In [ ]:
# Chained USING clauses (samples → sequencing_runs → qc_metrics)
query = """
SELECT sample_id, run_id, sequencer,
       mean_coverage, pass_filter
FROM   samples
INNER JOIN sequencing_runs USING (sample_id)
INNER JOIN qc_metrics      USING (run_id)
ORDER BY sample_id;
"""
df = run(query)
display(df)

---
## 11. Non-equi JOIN (Range / Inequality Join)

The `ON` clause can use **any boolean expression**, not just equality.
This is useful for range lookups, bucketing, and comparison joins.

In [ ]:
# Define coverage quality tiers in a CTE, then join qc_metrics on a range
query = """
WITH coverage_tiers (tier, min_cov, max_cov) AS (
    VALUES
        ('Low',    0.0,   50.0),
        ('Medium', 50.0, 100.0),
        ('High',  100.0, 999.0)
)
SELECT qm.run_id,
       qm.mean_coverage,
       ct.tier AS coverage_tier
FROM   qc_metrics qm
JOIN   coverage_tiers ct
          ON qm.mean_coverage >= ct.min_cov
         AND qm.mean_coverage <  ct.max_cov
ORDER BY qm.mean_coverage DESC;
"""
df = run(query)
display(df)

In [ ]:
# Comparison join: find all patient pairs where one is significantly older
# (age difference > 20 years) and they share a diagnosis
query = """
SELECT p1.patient_id AS younger_patient, p1.age AS younger_age,
       p2.patient_id AS older_patient,   p2.age AS older_age,
       p1.diagnosis,
       (p2.age - p1.age) AS age_gap
FROM   patients p1
JOIN   patients p2
          ON  p1.diagnosis  = p2.diagnosis
          AND p2.age - p1.age > 20
ORDER BY age_gap DESC;
"""
df = run(query)
print(f'Patient pairs sharing a diagnosis with >20 yr age gap: {len(df)}')
display(df)

---
## 12. Derived Table JOIN (Joining to a Subquery)

A subquery in the `FROM` clause acts as a temporary table (derived table or
inline view). It is evaluated first, and the result is joined like any
regular table. CTEs (`WITH` clauses) are the modern, readable alternative.

In [ ]:
# Derived table: pre-aggregate per-sequencer stats, then join to runs
query = """
SELECT sr.run_id, sr.sequencer, qm.mean_coverage,
       seq_avg.avg_coverage,
       ROUND(qm.mean_coverage - seq_avg.avg_coverage, 2) AS delta_from_avg
FROM   sequencing_runs sr
JOIN   qc_metrics qm ON sr.run_id = qm.run_id
JOIN   (
           SELECT sr2.sequencer,
                  AVG(qm2.mean_coverage) AS avg_coverage
           FROM   sequencing_runs sr2
           JOIN   qc_metrics qm2 ON sr2.run_id = qm2.run_id
           GROUP  BY sr2.sequencer
       ) seq_avg ON sr.sequencer = seq_avg.sequencer
ORDER BY sr.sequencer, delta_from_avg DESC;
"""
df = run(query)
print('Each run vs its sequencer-type average coverage:')
display(df)

In [ ]:
# Same query rewritten as a CTE (generally preferred for readability)
query = """
WITH seq_avg AS (
    SELECT sr.sequencer,
           AVG(qm.mean_coverage) AS avg_coverage
    FROM   sequencing_runs sr
    JOIN   qc_metrics qm ON sr.run_id = qm.run_id
    GROUP  BY sr.sequencer
)
SELECT sr.run_id, sr.sequencer,
       qm.mean_coverage,
       ROUND(seq_avg.avg_coverage, 2) AS seq_avg_coverage,
       ROUND(qm.mean_coverage - seq_avg.avg_coverage, 2) AS delta
FROM   sequencing_runs sr
JOIN   qc_metrics qm  ON sr.run_id    = qm.run_id
JOIN   seq_avg        ON sr.sequencer = seq_avg.sequencer
ORDER BY sr.sequencer, delta DESC;
"""
df = run(query)
print('CTE version — identical result:')
display(df)

---
## 13. Anti-JOIN

Returns rows from table A that have **no matching row** in table B.  
The pattern: `LEFT JOIN ... WHERE b.key IS NULL`.

Also expressible with `NOT EXISTS` or `NOT IN` (see below).

In [ ]:
# Anti-JOIN pattern 1: LEFT JOIN + IS NULL
# Find patients who have NO samples
query = """
SELECT p.patient_id, p.diagnosis, p.age
FROM   patients p
LEFT JOIN samples s ON p.patient_id = s.patient_id
WHERE  s.sample_id IS NULL
ORDER BY p.patient_id;
"""
df = run(query)
print('Patients with no samples (LEFT JOIN + IS NULL):')
display(df)

In [ ]:
# Anti-JOIN pattern 2: NOT EXISTS (often more readable)
query = """
SELECT p.patient_id, p.diagnosis, p.age
FROM   patients p
WHERE  NOT EXISTS (
    SELECT 1 FROM samples s WHERE s.patient_id = p.patient_id
)
ORDER BY p.patient_id;
"""
df = run(query)
print('Same result via NOT EXISTS:')
display(df)

In [ ]:
# Anti-JOIN: sequencing runs with no QC metrics entry
query = """
SELECT sr.run_id, sr.sample_id, sr.sequencer, sr.run_date
FROM   sequencing_runs sr
LEFT JOIN qc_metrics qm ON sr.run_id = qm.run_id
WHERE  qm.metric_id IS NULL
ORDER BY sr.run_id;
"""
df = run(query)
print('Runs missing QC metrics:')
display(df)

---
## 14. Semi-JOIN

Returns rows from table A where **at least one matching row exists** in table B,
without actually bringing columns from B into the result.

Typically written with `EXISTS` or `IN`.

In [ ]:
# Semi-JOIN: patients that have at least one high-coverage run (>= 100X)
query = """
SELECT p.patient_id, p.diagnosis, p.age
FROM   patients p
WHERE  EXISTS (
    SELECT 1
    FROM   samples         s
    JOIN   sequencing_runs sr ON s.sample_id = sr.sample_id
    JOIN   qc_metrics      qm ON sr.run_id   = qm.run_id
    WHERE  s.patient_id    = p.patient_id
      AND  qm.mean_coverage >= 100
)
ORDER BY p.patient_id;
"""
df = run(query)
print('Patients with ≥1 run having coverage ≥ 100X:')
display(df)

In [ ]:
# Semi-JOIN using IN (equivalent, simpler for single-column keys)
query = """
SELECT p.patient_id, p.diagnosis
FROM   patients p
WHERE  p.patient_id IN (
    SELECT s.patient_id
    FROM   samples         s
    JOIN   sequencing_runs sr ON s.sample_id  = sr.sample_id
    JOIN   qc_metrics      qm ON sr.run_id    = qm.run_id
    WHERE  qm.mean_coverage >= 100
)
ORDER BY p.patient_id;
"""
df = run(query)
print('Same result via IN:')
display(df)

In [ ]:
# Semi-JOIN via DISTINCT JOIN (less recommended — can return duplicates if not careful)
query = """
SELECT DISTINCT p.patient_id, p.diagnosis
FROM   patients p
JOIN   samples         s  ON p.patient_id = s.patient_id
JOIN   sequencing_runs sr ON s.sample_id  = sr.sample_id
JOIN   qc_metrics      qm ON sr.run_id    = qm.run_id
WHERE  qm.mean_coverage >= 100
ORDER BY p.patient_id;
"""
df = run(query)
print('Same result via DISTINCT JOIN:')
display(df)

---
## 15. NATURAL JOIN

Automatically joins on **all columns with the same name** in both tables.
No `ON` clause needed, but can be brittle if schema changes add accidental
column name collisions. Use with caution in production code.

In [ ]:
# NATURAL JOIN: sequencing_runs × qc_metrics share only 'run_id'
# PostgreSQL will join on run_id automatically.
query = """
SELECT run_id, sequencer, run_date,
       mean_coverage, mean_quality_score, pass_filter
FROM   sequencing_runs
NATURAL JOIN qc_metrics
ORDER BY run_id;
"""
df = run(query)
print('NATURAL JOIN on run_id (automatically detected):')
display(df)

In [ ]:
# Demonstrate why NATURAL JOIN can be risky:
# samples and sequencing_runs share BOTH sample_id AND patient_id? No —
# but they share 'sample_id'. Let's see what NATURAL JOIN does here.
query = """
SELECT sample_id, patient_id, sample_type,
       run_id, sequencer, run_date
FROM   samples
NATURAL JOIN sequencing_runs
ORDER BY sample_id;
"""
df = run(query)
print('NATURAL JOIN samples × sequencing_runs (joins on sample_id):')
display(df)

---
## 16. Quick Reference Cheat Sheet

```sql
-- INNER JOIN: only matched rows
SELECT * FROM a INNER JOIN b ON a.id = b.id;

-- LEFT JOIN: all rows from a, NULLs from b when no match
SELECT * FROM a LEFT JOIN b ON a.id = b.id;

-- RIGHT JOIN: all rows from b, NULLs from a when no match
SELECT * FROM a RIGHT JOIN b ON a.id = b.id;

-- FULL OUTER JOIN: all rows from both, NULLs on either side
SELECT * FROM a FULL OUTER JOIN b ON a.id = b.id;

-- CROSS JOIN: every combination (M × N rows)
SELECT * FROM a CROSS JOIN b;

-- SELF JOIN: table joined to itself
SELECT * FROM t AS t1 JOIN t AS t2 ON t1.col = t2.col AND t1.id < t2.id;

-- JOIN with USING: shorthand when key column names match
SELECT * FROM a JOIN b USING (shared_col);

-- Non-equi JOIN: join on a range/inequality
SELECT * FROM a JOIN b ON a.val BETWEEN b.lo AND b.hi;

-- Derived table JOIN
SELECT * FROM a JOIN (SELECT ...) sub ON a.id = sub.id;

-- Anti-JOIN: rows in a with no match in b
SELECT * FROM a LEFT JOIN b ON a.id = b.id WHERE b.id IS NULL;
-- or:
SELECT * FROM a WHERE NOT EXISTS (SELECT 1 FROM b WHERE b.id = a.id);

-- Semi-JOIN: rows in a that have a match in b (no b columns returned)
SELECT * FROM a WHERE EXISTS (SELECT 1 FROM b WHERE b.id = a.id);
-- or:
SELECT * FROM a WHERE a.id IN (SELECT id FROM b);

-- NATURAL JOIN: auto-join on identically named columns (use with caution)
SELECT * FROM a NATURAL JOIN b;
```

### Key Rules to Remember

| Situation | Best Choice |
|-----------|-------------|
| Need rows only where both sides match | `INNER JOIN` |
| Need all rows from the primary/driving table | `LEFT JOIN` |
| Check for missing / unmatched rows | `LEFT JOIN … WHERE b.col IS NULL` (anti-join) |
| Check existence without pulling columns | `EXISTS` (semi-join) |
| All combinations for a reporting matrix | `CROSS JOIN` |
| Compare rows within a single table | `SELF JOIN` |
| Join key has the same name in both tables | `USING (col)` — cleaner than `ON a.col = b.col` |

---
## 17. Cleanup — Remove Orphan Rows

Run this cell to remove the demonstration orphan rows added in Section 2,
restoring the database to its original state.

In [ ]:
conn = get_conn()
cur  = conn.cursor()

cur.execute("DELETE FROM sequencing_runs WHERE run_id     = 'RUN9999';")
cur.execute("DELETE FROM samples           WHERE sample_id  = 'S9901';")
cur.execute("DELETE FROM patients          WHERE patient_id IN ('PT9901','PT9902');")

conn.commit()
cur.close()
conn.close()

print('Orphan rows removed. Database restored to original state.')